In [ ]:
import requests
import pandas as pd
from datetime import datetime

API_URL = "https://api.hyperliquid.xyz/info"

tickers = [
    'ATOM','REQ','CRV','MAVIA','SAGA','NEAR','MORPHO','MANTA','MOVE','XAI',
    'ETC','DOGE','SOPH','CELO','MAV','POPCAT','SCR','COMP','GMT','SOL','IMX',
    'JUP','RUNE','LAUNCHCOIN','UMA','TRB','USTC','AIXBT','IOTA','VIRTUAL',
    'ALGO','GMX','ANIME','BCH','BIO','BSV','NXPC','MOODENG','TNSR','HBAR',
    'SNX','ZEREBRO','HYPER','SAND','BERA','PURR','GAS','LDO','ONDO','DYDX',
    'FTT','TON','EIGEN','LTC','BLAST','AI16Z','OMNI','AAVE','OGN','SUI',
    'MEME','FXS','NEIROETH','NIL','CFX','ME','XRP','TIA','BNB','NOT','IP',
    'OM','TAO','OP','CAKE','AVAX','kPEPE','GALA','MNT','BOME','SUPER','SEI',
    'VINE','KAS','BABY','STX','S','FARTCOIN','STG','RENDER','ENA','LINK',
    'ARB','ARK','BIGTIME','BTC','ETH','RSR','kDOGS','BRETT','BANANA','XLM',
    'INJ','ENS','AR','DOT','SPX','ETHFI','PAXG','kLUNC','GOAT','kSHIB','FIL',
    'MEW','STRK','TRX','ZK','KAITO','PENGU','kBONK','VVV','ORDI','INIT','APT',
    'REZ','LAYER','ZEN','SUSHI','kFLOKI','ADA','kNEIRO','PEOPLE','ZORA',
    'PENDLE','APE','HYPE','FET','CHILLGUY','MELANIA','GRIFFAIN','PNUT','DOOD',
    'WIF','ACE','ZETA','TRUMP','NEO','JTO','YGG','ZRO','PROMPT','WLD','W',
    'MERL','BLUR','UNI','DYM','MINA','MKR','POLYX','POL','IO','TURBO','PYTH',
    'USUAL','GRASS','ALT','HMSTR','WCT','SYRUP','RESOLV','PROVE','YZY','WLFI',
    'TST','PUMP','LINEA','SKY','ASTER','0G','STBL','AVNT','XPL','ZEC','ICP'
]

def fetch_funding(coin, start_ms, end_ms):
    payload = {
        "type": "fundingHistory",
        "req": {
            "coin": coin,
            "startTime": start_ms,
            "endTime": end_ms
        }
    }
    r = requests.post(API_URL, json=payload)
    try:
        r.raise_for_status()
    except:
        return pd.DataFrame()

    data = r.json()
    if not isinstance(data, list) or len(data) == 0:
        return pd.DataFrame()

    df = pd.DataFrame(data)
    df["time"] = pd.to_datetime(df["time"], unit="ms")

    # Funding payouts (00:00, 08:00, 16:00 UTC)
    df_8h = df[df["time"].dt.hour.isin([0, 8, 16])]
    return df_8h


# -----------------------------------------------------------------------------
# MAIN SCRIPT: DOWNLOAD FUNDING PAYOUTS FOR ALL COINS
# -----------------------------------------------------------------------------

start = datetime(2025, 6, 5)
now = datetime.utcnow()

start_ms = int(start.timestamp() * 1000)
end_ms = int(now.timestamp() * 1000)

funding_rows = []

for ticker in tickers:
    print(f"Downloading 8h funding payouts for {ticker}...")
    df_f = fetch_funding(ticker, start_ms, end_ms)

    if df_f.empty:
        print(f"⚠️ No funding for {ticker} — skipped")
        continue

    for _, row in df_f.iterrows():
        funding_rows.append({
            "perp": ticker,
            "time": row["time"],
            "fundingRate": row["fundingRate"]
        })

funding_df = pd.DataFrame(funding_rows)
funding_df = funding_df.sort_values(["perp", "time"])

funding_df.to_csv("all_perps_8h_funding_payouts.csv", index=False)

print("\n✔️ Saved → all_perps_8h_funding_payouts.csv")
